# Experiment: Taobao Attributed Order Performance Analysis

## Objective

Reproduce aggregate product-tag, promotion-slot, and order-risk findings from the processed attributed-order tables.

Success criteria:
- Reconcile line-level and order-level attributed payment amount.
- Keep incomplete months out of ordinary month-over-month comparisons.
- Treat recommendations as test or review hypotheses, not proof of advertising efficiency or causality.


In [ ]:
# Setup and reproducibility
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
ANALYSIS_DIR = PROJECT_ROOT / 'outputs' / 'analysis'
assert (PROJECT_ROOT / 'src' / 'analyze_performance.py').exists()
PROJECT_ROOT


## Analysis plan

- Use `付款金额` only as attributed payment amount.
- Use distinct `淘宝订单编号` for order counts and risks.
- Define the primary invalid-order rate as invalid orders divided by invalid plus non-invalid orders; show mixed-status orders separately.
- Require at least 30 attributed orders and 7 active days for a promotion to enter the main scale-risk comparison.


In [ ]:
# Load the generated aggregate tables
lines = pd.read_csv(PROCESSED_DIR / 'order_lines_clean.csv', dtype={'商品ID': 'string', '淘宝订单编号': 'string'})
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv', dtype={'淘宝订单编号': 'string'})
monthly = pd.read_csv(ANALYSIS_DIR / 'monthly_performance.csv')
tags = pd.read_csv(ANALYSIS_DIR / 'tag_performance.csv')
promotions = pd.read_csv(ANALYSIS_DIR / 'promotion_performance.csv', dtype={'推广位ID': 'string'})
summary = json.loads((ANALYSIS_DIR / 'analysis_summary.json').read_text(encoding='utf-8'))
summary['scope']


## Reconciliation checks

The following assertions are deliberately small and auditable. They verify aggregation grain and the period-comparison rule; they do not establish commercial causality.


In [ ]:
assert len(lines) == summary['scope']['order_line_count']
assert len(orders) == summary['scope']['attributed_order_count']
assert round(lines['付款金额'].sum(), 2) == round(orders['attributed_payment_amount'].sum(), 2)
assert round(tags['attributed_payment_amount'].sum(), 2) == round(lines['付款金额'].sum(), 2)
assert monthly.loc[~monthly['is_complete_month'], 'payment_amount_mom'].isna().all()
assert promotions.loc[promotions['is_main_comparison_eligible'], 'attributed_order_count'].ge(30).all()
assert promotions.loc[promotions['is_main_comparison_eligible'], 'active_days'].ge(7).all()
monthly[['click_month', 'is_complete_month', 'attributed_payment_amount', 'invalid_order_rate', 'zero_payment_line_rate', 'payment_amount_mom']]


## Decision-oriented outputs

The outputs below rank observed scale and risk. They are candidates for investigation or a narrowly scoped test only; they are not an ROI ranking.


In [ ]:
high_risk_tags = tags.loc[tags['scale_risk_quadrant'].eq('high_scale_high_risk')].sort_values('attributed_payment_amount', ascending=False)
high_risk_promotions = promotions.loc[promotions['scale_risk_quadrant'].eq('high_scale_high_risk')].sort_values('attributed_payment_amount', ascending=False)
display(high_risk_tags[['一级标签', '二级标签', 'attributed_payment_amount', 'attributed_order_count', 'invalid_order_rate', 'zero_payment_line_rate']].head(10))
display(high_risk_promotions[['推广位ID', '推广位名称', 'attributed_payment_amount', 'attributed_order_count', 'active_days', 'invalid_order_rate']].head(10))


## Next steps

- Review the May zero-payment spike before drawing product or promotion conclusions from that month.
- Diagnose high-scale, high-risk tags and promotion slots using order-status and zero-payment splits.
- Collect exposure, total-click, cost, and settlement data before evaluating advertising efficiency, ROI, or causal lift.
